# Marketing Mix Modeling — Fitting Notebook

This notebook fits a **Bayesian MMM** on synthetic weekly marketing data and exports results as CSVs for the Streamlit dashboard.

**Workflow:**
1. Install PyMC-Marketing (Cell 1)
2. Generate the same synthetic dataset used in `app.py` (Cell 3)
3. Fit a Bayesian MMM with adstock + saturation (Cell 4)
4. Extract posterior estimates and save 3 CSV files (Cell 5)
5. Download the CSVs and place them in the `results/` folder next to `app.py`


In [ ]:
# Cell 1 — Install pymc-marketing (~3 min on Colab, runtime will restart automatically)
!pip install pymc-marketing -q

## Why PyMC-Marketing?

**PyMC-Marketing** is a Bayesian MMM library built on PyMC. Unlike frequentist approaches (e.g. Ridge regression), it:
- Quantifies **uncertainty** — every coefficient has a full posterior distribution, not just a point estimate
- Returns **credible intervals** (89% HDI) on ROI and channel contributions
- Incorporates domain priors (e.g. adstock decay must be between 0 and 1)

The trade-off is speed: MCMC sampling takes 5–15 minutes vs. milliseconds for Ridge.


In [ ]:
# Cell 2 — Imports
import os
import numpy as np
import pandas as pd

# pymc-marketing >= 0.10 moved MMM to the multidimensional module
try:
    from pymc_marketing.mmm.multidimensional import MMM
    print('Using pymc_marketing.mmm.multidimensional.MMM')
except ImportError:
    from pymc_marketing.mmm import MMM
    print('Using pymc_marketing.mmm.MMM (legacy)')

from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
print('Imports OK')

## The Dataset — Synthetic but Realistic

The data is generated with **known ground-truth parameters**, which gives this demo a unique advantage: we can verify how well the model recovers the true ROI.

| Channel | True ROI | Adstock decay | Half-saturation |
|---------|----------|---------------|-----------------|
| TV | £0.55 | 0.70 (high carry-over) | £30k |
| Digital | £1.10 | 0.40 (mostly immediate) | £18k |
| Social | £0.90 | 0.30 | £9k |
| OOH | £0.40 | 0.60 | £6k |

No real-world MMM project can show this comparison table. Synthetic data lets us validate the methodology scientifically.


In [ ]:
# Cell 3 — Reproduce the exact same synthetic dataset used in app.py (seed=42)
def make_dataset(seed=42, n_weeks=104):
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2022-01-03', periods=n_weeks, freq='W-MON')
    t     = np.arange(n_weeks)

    channels = {
        'tv_spend':      (40_000, 15_000),
        'digital_spend': (25_000,  8_000),
        'social_spend':  (12_000,  4_000),
        'ooh_spend':     ( 8_000,  3_000),
    }

    def adstock(x, decay):
        out = np.zeros_like(x, dtype=float)
        out[0] = x[0]
        for i in range(1, len(x)):
            out[i] = x[i] + decay * out[i - 1]
        return out

    def hill(x, half_sat, slope=2.0):
        return x**slope / (half_sat**slope + x**slope)

    roi      = {'tv_spend': 0.55, 'digital_spend': 1.10, 'social_spend': 0.90, 'ooh_spend': 0.40}
    decay    = {'tv_spend': 0.70, 'digital_spend': 0.40, 'social_spend': 0.30, 'ooh_spend': 0.60}
    half_sat = {'tv_spend': 30_000, 'digital_spend': 18_000, 'social_spend': 9_000, 'ooh_spend': 6_000}

    seasonality = (
        1 + 0.20 * np.sin(2 * np.pi * t / 52)
        + 0.08 * np.sin(4 * np.pi * t / 52 + 0.5)
    )
    trend = 50_000 + 80 * t

    df = pd.DataFrame({'date': dates})
    media_contrib = np.zeros(n_weeks)

    for ch, (mean_spend, std_spend) in channels.items():
        raw = rng.normal(mean_spend, std_spend, n_weeks).clip(0)
        raw *= (0.85 + 0.30 * np.abs(np.sin(np.pi * t / 52 + rng.uniform(0, np.pi))))
        df[ch] = raw.round(0)
        adstocked = adstock(raw, decay[ch])
        saturated = hill(adstocked, half_sat[ch]) * half_sat[ch]
        media_contrib += roi[ch] * saturated

    df['sales'] = (
        trend * seasonality + media_contrib + rng.normal(0, 4_000, n_weeks)
    ).round(0).clip(0)

    return df


df           = make_dataset()
date_col     = 'date'
target_col   = 'sales'
channel_cols = ['tv_spend', 'digital_spend', 'social_spend', 'ooh_spend']

print(df.shape)
df.head()

## The MMM Model — Adstock + Saturation + Trend + Seasonality

The model decomposes observed sales as:

```
sales(t) = baseline(t) + sum_ch [ ROI_ch * saturate(adstock(spend_ch(t))) ] + noise
```

- **GeometricAdstock(l_max=8)** — spend at week *t* carries over geometrically for up to 8 weeks. TV has high carry-over (decay 0.70); Digital is more immediate (0.40).
- **LogisticSaturation()** — diminishing returns: extra spend yields less incremental response as budgets grow.
- **yearly_seasonality=2** — Fourier terms to capture annual cycles.
- **NUTS sampler** — samples the full posterior distribution of all parameters.

> **Note on model spec:** The data was generated with a Hill saturation curve; PyMC-Marketing uses LogisticSaturation. This mismatch causes some parameter recovery error — a real-world lesson about the importance of model specification.


In [ ]:
# Cell 4 — Fit PyMC-Marketing MMM  (GeometricAdstock + LogisticSaturation)
# Expected runtime: 5-15 minutes on Colab CPU

# Prepare X (features) and y (target) separately for the multidimensional MMM
X = df[[date_col] + channel_cols].copy()
y = df[target_col].astype(float)

mmm = MMM(
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    date_column=date_col,
    channel_columns=channel_cols,
    target_column=target_col,   # required by multidimensional API
    yearly_seasonality=2,
)

print('Sampling posterior — please wait...')
idata = mmm.fit(
    X=X, y=y,
    draws=500, tune=500, chains=2,
    target_accept=0.95,
    progressbar=True,
)

# Explicitly sample posterior predictive
print('Sampling posterior predictive...')
try:
    if not hasattr(idata, 'posterior_predictive'):
        mmm.sample_posterior_predictive(X, extend_idata=True)
        print('Posterior predictive sampled.')
    else:
        print('Posterior predictive already in idata.')
except Exception as _e:
    print(f'Could not sample posterior predictive ({_e}); Cell 5 will use contribution fallback.')

print('Sampling complete!')

In [ ]:
# Cell 4b — Diagnose posterior variable shapes
total_orig = idata.posterior["total_media_contribution_original_scale"]
ch_norm    = idata.posterior["channel_contribution"]

print("channel_contribution       dims:", ch_norm.dims, "shape:", ch_norm.shape)
print("total_media_orig           dims:", total_orig.dims, "shape:", total_orig.shape)
print()
print(f"total_media_orig  mean per sample: {float(total_orig.mean()):.2f}")
print(f"channel_contribution sum (all)  : {float(ch_norm.sum(['date','channel']).mean()):.4f}")
print()
print(f"Total actual sales (sum): {df['sales'].sum():,.0f}")
print(f"Total actual sales (mean/week):  {df['sales'].mean():,.0f}")

## Extracting Results — 3 CSV Files

After sampling, we extract three files:

| File | Contents |
|------|----------|
| `mmm_model_results.csv` | Per-channel: contribution (£), ROI, spend, baseline |
| `mmm_fitted_values.csv` | Weekly: posterior mean fitted sales + 89% credible interval |
| `mmm_contrib_hdi.csv` | Per-channel contribution mean + 89% HDI |

Download these 3 files and place them in the `results/` folder next to `app.py`. The dashboard auto-loads them on startup.


In [ ]:
# Cell 5 — Extract results and save 3 CSV files
os.makedirs('results', exist_ok=True)

# ── Channel contributions (original scale) ────────────────────────────────
# total_media_contribution_original_scale is (chain, draw) — total £ across all dates & channels
# channel_contribution is (chain, draw, date, channel) — in normalized scale
# → compute each channel's fraction of total normalized, then apply to £ total
try:
    contributions_norm = idata.posterior["channel_contribution"]        # (chain, draw, date, channel)
    total_media_orig   = idata.posterior["total_media_contribution_original_scale"]  # (chain, draw)

    ch_dim   = contributions_norm.dims[-1]
    chan_names = list(contributions_norm.coords[ch_dim].values)
    date_dim = [d for d in contributions_norm.dims if d not in ('chain', 'draw', ch_dim)][0]

    # Per-channel fraction of total normalized contribution (sum over dates per channel)
    total_by_ch_norm = contributions_norm.sum(date_dim)          # (chain, draw, channel)
    total_all_norm   = total_by_ch_norm.sum(ch_dim)              # (chain, draw)
    ch_fraction      = total_by_ch_norm / total_all_norm         # (chain, draw, channel)

    # Scale fractions by original-£ total — broadcasts (chain, draw) × (chain, draw, channel)
    per_channel_orig = ch_fraction * total_media_orig            # (chain, draw, channel)
    samples_2d = per_channel_orig.values.reshape(-1, len(chan_names))

    mean_contribs  = samples_2d.mean(axis=0)
    lower_contribs = np.percentile(samples_2d, 5.5, axis=0)
    upper_contribs = np.percentile(samples_2d, 94.5, axis=0)
    print(f"Total media (£): {float(total_media_orig.mean()):,.0f}  |  total sales: {df['sales'].sum():,.0f}")
    print(f"Mean contributions (£): {dict(zip(chan_names, mean_contribs.round(0)))}")
except Exception as e:
    print(f"Error extracting contributions: {e}")
    chan_names = channel_cols
    mean_contribs = np.zeros(len(chan_names))
    lower_contribs = np.zeros(len(chan_names))
    upper_contribs = np.zeros(len(chan_names))

# ── Fitted values (posterior predictive) ──────────────────────────────────
try:
    pp    = idata.posterior_predictive
    y_key = next((k for k in pp.data_vars if k in ('y', 'mu', 'likelihood', 'obs')), None)
    if y_key is None:
        raise KeyError('No likelihood key in posterior_predictive')
    y_samples = pp[y_key].values.reshape(-1, len(df))
    print(f'Using posterior_predictive["{y_key}"] for fitted values')
except Exception as err:
    print(f'Posterior predictive unavailable ({err}); using mean target as fallback')
    y_samples = np.tile(df[target_col].values, (1000, 1))

y_mean  = y_samples.mean(axis=0)
y_lower = np.percentile(y_samples, 5.5, axis=0)
y_upper = np.percentile(y_samples, 94.5, axis=0)

# ── Baseline ──────────────────────────────────────────────────────────────
total_spend    = df[channel_cols].sum()
baseline_total = float(y_mean.sum()) - float(mean_contribs.sum())

# ── 1. mmm_model_results.csv ────────────────────────────────────────────
rows = []
for i, ch in enumerate(chan_names):
    spend   = float(total_spend.get(ch, 0))
    contrib = float(mean_contribs[i])
    rows.append({
        'channel':      ch,
        'contribution': round(contrib, 0),
        'roi':          round(contrib / spend, 4) if spend > 0 else 0.0,
        'spend':        round(spend, 0),
        'baseline':     round(baseline_total, 0),
    })
pd.DataFrame(rows).to_csv('results/mmm_model_results.csv', index=False)
print('Saved results/mmm_model_results.csv')

# ── 2. mmm_fitted_values.csv ────────────────────────────────────────────
pd.DataFrame({
    'date':         df['date'].astype(str),
    'fitted_sales': y_mean.round(0),
    'lower_89':     y_lower.round(0),
    'upper_89':     y_upper.round(0),
}).to_csv('results/mmm_fitted_values.csv', index=False)
print('Saved results/mmm_fitted_values.csv')

# ── 3. mmm_contrib_hdi.csv ──────────────────────────────────────────────
pd.DataFrame({
    'channel':  chan_names,
    'mean':     mean_contribs.round(0),
    'lower_89': lower_contribs.round(0),
    'upper_89': upper_contribs.round(0),
}).to_csv('results/mmm_contrib_hdi.csv', index=False)
print('Saved results/mmm_contrib_hdi.csv')

print('\nAll 3 CSVs ready in results/')

In [ ]:
# Cell 6a — Download mmm_model_results.csv
from google.colab import files
files.download('results/mmm_model_results.csv')

In [ ]:
# Cell 6b — Download mmm_fitted_values.csv
from google.colab import files
files.download('results/mmm_fitted_values.csv')

In [ ]:
# Cell 6c — Download mmm_contrib_hdi.csv
from google.colab import files
files.download('results/mmm_contrib_hdi.csv')